## 多向量检索器

每个文档存储多个向量通常是有益的。在许多用例中，这是有益的。 LangChain 有一个基础 MultiVectorRetriever ，这使得查询此类设置变得容易。很多复杂性在于如何为每个文档创建多个向量。本笔记本涵盖了创建这些向量和使用 MultiVectorRetriever 的一些常见方法。

为每个文档创建多个向量的方法包括：
- 较小的块：将文档分割成较小的块，然后嵌入这些块（这是 ParentDocumentRetriever）。
- 摘要：为每个文档创建摘要，将其与文档一起嵌入（或代替文档）
- 假设性问题：创建每个文档都适合回答的假设性问题，将这些问题与文档一起嵌入（或代替文档）。

请注意，这还启用了另一种添加嵌入的方法 - 手动。这很棒，因为您可以显式添加导致文档恢复的问题或查询，从而为您提供更多控制权。

In [1]:
from langchain.retrievers.multi_vector import MultiVectorRetriever

In [2]:
from langchain.storage import InMemoryByteStore
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
loaders = [
    TextLoader("../02-agent/txt/faq-4359.txt",encoding="utf-8"),
    TextLoader("../02-agent/txt/faq-7923.txt",encoding="utf-8"),
]
loaders

In [4]:
docs = []
for loader in loaders:
    docs.extend(loader.load())

docs

[Document(metadata={'source': '../02-agent/txt/faq-4359.txt'}, page_content='一、什么是0分期利息\n\n您好，“0分期利息”是指买家使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付分期利息的功能，分期利息由华为商城承担。\n\n注：自2023年起，商城将相关宣传将“免息”调整为“0分期利息”，主要基于中国银保监会、中国人民银行《关于进一步促进信用卡业务规范健康发展的通知》（银保监规〔2022〕13号），要求“银行业金融机构应当在分期业务合同（协议）首页和业务办理页面以明显方式展示分期业务可能产生的所有息费项目、年化利率水平和息费计算方式。向客户展示分期业务收取的资金使用成本时，应当统一采用利息形式，并明确相应的计息规则，不得采用手续费等形式，法律法规另有规定的除外。”\n\n\n\n二、可以参与0分期利息活动的商品\n\n商城目前仅支持部分单品参与0分期利息，若多商品（含不支持0分期利息）合并支付则不支持0分期利息，以支付页面为准，后续会逐渐开放更多商品，敬请关注。\n\n\n\n三、确认订单分期成功\n\n订单提交成功后在支付方式页面选择分期支付，点选显示0分期利息的支付方式及具体0分期利息期数后，完成支付。\n\n\n\n\n\n四、订单中有多个商品，其中有商品支持0分期利息，为什么提交后却没有0分期利息？\n\n0分期利息商品不能和其它商品一起购买，如果和其他商品购买而不能享用0分期利息，建议取消原来的订单，重新购买时把0分期利息商品和其他商品分开单独购买；且0分期利息的分期数是固定的，如6期0分期利息，并不是3/6/12都提供0分期利息的。\n\n\n\n五、小程序是否支持0分期利息？\n\n华为商城小程序暂不支持0分期利息。'),
 Document(metadata={'source': '../02-agent/txt/faq-7923.txt'}, page_content='众测活动\n\n整体介绍：\n\n一、活动定义：众测是以低价试销的形式，通过收集评价、销量等方法，用于测试市场对新商品的反应，用于及时优化销售策略和引导商家改进。\n\n二、优势：众测价通常比较优惠，以不高于大促促销价为原则，最终以与物权方谈判结果为准。\n\n三、适用范围：华为

In [5]:
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000)
# docs = text_splitter.split_documents(docs)

In [6]:
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings
# embeddings_path = "D:\\ai\\download\\bge-large-zh-v1.5"
embeddings_path = "/home/vincent/.lmstudio/models/bge-small-zh-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=embeddings_path)

/tmp/ipykernel_5521/3172452146.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embeddings_path)
/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce MX150 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions

In [7]:
# 用于索引子块的向量存储
vectorstore = Chroma(
    collection_name="full_documents", embedding_function=embeddings
)

/tmp/ipykernel_5521/111971831.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [8]:
# 父文档的存储层
store = InMemoryByteStore()
id_key = "doc_id"
# 检索器（空启动）
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

In [9]:
import uuid

doc_ids = [str(uuid.uuid4()) for _ in docs]
doc_ids

['f233691b-9c61-4700-9904-4a956ec43bae',
 '1c15ebd9-7b5b-4042-883e-a124deee0021']

In [10]:
from langchain_text_splitters import CharacterTextSplitter
# 用于创建较小块的分割器
child_text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=10,
    length_function=len,
    is_separator_regex=False,
)

In [11]:
sub_docs = []
for i, doc in enumerate(docs):
    _id = doc_ids[i]
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    sub_docs.extend(_sub_docs)
sub_docs

Created a chunk of size 224, which is longer than the specified 100
Created a chunk of size 117, which is longer than the specified 100


[Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f233691b-9c61-4700-9904-4a956ec43bae'}, page_content='一、什么是0分期利息\n\n您好，“0分期利息”是指买家使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付分期利息的功能，分期利息由华为商城承担。'),
 Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f233691b-9c61-4700-9904-4a956ec43bae'}, page_content='注：自2023年起，商城将相关宣传将“免息”调整为“0分期利息”，主要基于中国银保监会、中国人民银行《关于进一步促进信用卡业务规范健康发展的通知》（银保监规〔2022〕13号），要求“银行业金融机构应当在分期业务合同（协议）首页和业务办理页面以明显方式展示分期业务可能产生的所有息费项目、年化利率水平和息费计算方式。向客户展示分期业务收取的资金使用成本时，应当统一采用利息形式，并明确相应的计息规则，不得采用手续费等形式，法律法规另有规定的除外。”'),
 Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f233691b-9c61-4700-9904-4a956ec43bae'}, page_content='二、可以参与0分期利息活动的商品\n\n商城目前仅支持部分单品参与0分期利息，若多商品（含不支持0分期利息）合并支付则不支持0分期利息，以支付页面为准，后续会逐渐开放更多商品，敬请关注。'),
 Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f233691b-9c61-4700-9904-4a956ec43bae'}, page_content='三、确认订单分期成功\n\n订单提交成功后在支付方式页面选择分期支付，点选显示0分期利息的支付方式及具体0分期利息期数后，完成支付。'),
 Document(

In [12]:
#使用一个名为retriever的对象来向一个向量存储（vectorstore）中添加文档，
#并且使用一个文档存储（docstore）来设置文档ID与文档内容之间的映射。
#这两个属性分别用于存储文档的向量化表示和文档的内容。
retriever.vectorstore.add_documents(sub_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

AcceleratorError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Vectorstore 单独检索小块
retriever.vectorstore.similarity_search("众测商品多久发货呢？")[0]

Document(metadata={'source': '../02-agent/txt/faq-7923.txt', 'doc_id': 'b6df11d1-be25-4d5d-9773-4edec10cb2d4'}, page_content='3、众测活动时间：\n\n一款产品众测时间通常为10-20天（热销的商品可能会延期5-10天）。\n\n \n\n常见问题：\n\n1、众测商品的来源?')

# 摘要总结
通常，摘要可能能够更准确地提炼出某个块的内容，从而实现更好的检索。在这里，我们展示如何创建摘要，然后嵌入它们。



In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)


In [ ]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("总结下面的文档:\n\n{doc}")
    | model
    | StrOutputParser()
)

In [ ]:
docs = []
for loader in loaders:
    docs.extend(loader.load())

docs

[Document(metadata={'source': '../02-agent/txt/faq-4359.txt'}, page_content='一、什么是0分期利息\n\n您好，“0分期利息”是指买家使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付分期利息的功能，分期利息由华为商城承担。\n\n注：自2023年起，商城将相关宣传将“免息”调整为“0分期利息”，主要基于中国银保监会、中国人民银行《关于进一步促进信用卡业务规范健康发展的通知》（银保监规〔2022〕13号），要求“银行业金融机构应当在分期业务合同（协议）首页和业务办理页面以明显方式展示分期业务可能产生的所有息费项目、年化利率水平和息费计算方式。向客户展示分期业务收取的资金使用成本时，应当统一采用利息形式，并明确相应的计息规则，不得采用手续费等形式，法律法规另有规定的除外。”\n\n\n\n二、可以参与0分期利息活动的商品\n\n商城目前仅支持部分单品参与0分期利息，若多商品（含不支持0分期利息）合并支付则不支持0分期利息，以支付页面为准，后续会逐渐开放更多商品，敬请关注。\n\n\n\n三、确认订单分期成功\n\n订单提交成功后在支付方式页面选择分期支付，点选显示0分期利息的支付方式及具体0分期利息期数后，完成支付。\n\n\n\n\n\n四、订单中有多个商品，其中有商品支持0分期利息，为什么提交后却没有0分期利息？\n\n0分期利息商品不能和其它商品一起购买，如果和其他商品购买而不能享用0分期利息，建议取消原来的订单，重新购买时把0分期利息商品和其他商品分开单独购买；且0分期利息的分期数是固定的，如6期0分期利息，并不是3/6/12都提供0分期利息的。\n\n\n\n五、小程序是否支持0分期利息？\n\n华为商城小程序暂不支持0分期利息。'),
 Document(metadata={'source': '../02-agent/txt/faq-7923.txt'}, page_content='众测活动\n\n整体介绍：\n\n一、活动定义：众测是以低价试销的形式，通过收集评价、销量等方法，用于测试市场对新商品的反应，用于及时优化销售策略和引导商家改进。\n\n二、优势：众测价通常比较优惠，以不高于大促促销价为原则，最终以与物权方谈判结果为准。\n\n三、适用范围：华为

In [ ]:
summaries = chain.batch(docs, {"max_concurrency": 5})
summaries

['<think>\n好的，我需要总结用户提供的文档内容。首先，用户给了一个关于“0分期利息”的文档，分为五个部分。我要确保每个要点都被准确概括。\n\n第一部分解释了什么是0分期利息，说明它不需要支付利息，由华为商城承担，并提到调整后的宣传名称。这里的关键点是定义、功能和政策变化。\n\n第二部分提到了可以参与活动的商品，指出仅支持部分单品，合并支付不支持，后续会开放更多商品。需要强调支持的商品类型和规则。\n\n第三部分讲确认订单分期成功的方法，即选择分期支付并显示0分期利息信息。这里要明确步骤和操作方式。\n\n第四部分解释了为什么提交后没有0分期利息，指出不能与其他商品同时购买，并且分期数固定。需要说明限制条件和影响因素。\n\n第五部分询问小程序是否支持0分期利息，回答是华为商城小程序暂不支持。最后总结所有要点，确保信息完整无遗漏。\n\n现在检查是否有重复或冗余的信息，比如政策变化和活动规则可能需要合并。同时，保持语言简洁明了，符合用户要求的总结格式。\n</think>\n\n以下是文档内容的总结：\n\n**一、0分期利息定义与调整**  \n“0分期利息”指通过花呗、招行掌上生活等平台购物时无需支付利息的功能，由华为商城承担。自2023年起，宣传名称从“免息”改为“0分期利息”，依据中国银保监会政策要求，明确展示分期费用项目及计费规则。\n\n**二、支持商品与活动规则**  \n商城仅支持部分单品参与0分期利息活动，合并支付不支持。后续将逐步开放更多商品，以支付页面为准。\n\n**三、确认订单分期成功**  \n提交订单后选择分期支付并显示0分期利息信息后完成支付。\n\n**四、0分期利息限制条件**  \n若订单包含支持0分期的商品，且与其他商品同时购买，则无法享受0分期利息。需将0分期商品与非该商品分开单独购买，并明确分期数（如6期）。\n\n**五、小程序功能说明**  \n华为商城小程序暂不提供0分期利息服务。',
 '<think>\n好的，用户让我总结提供的文档内容。首先，我需要通读整个文档，理解每个部分的信息。\n\n文档分为整体介绍、优势、适用范围、参与方式和常见问题几个部分。整体介绍里提到了众测的定义、优势以及适用范围。然后是参与方式，包括入口位置、上新频次和活动时间。常见问题也列出了来源、价格优惠、质量稳定性、发货时间、支

In [ ]:
# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="summaries", embedding_function=embeddings)
# The storage layer for the parent documents
store = InMemoryByteStore()
id_key = "doc_id"
# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

In [ ]:
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]
summary_docs

[Document(metadata={'doc_id': 'f274cb58-67f5-46ba-add6-8e4804f79f13'}, page_content='<think>\n好的，我需要总结用户提供的文档内容。首先，用户给了一个关于“0分期利息”的文档，分为五个部分。我要确保每个要点都被准确概括。\n\n第一部分解释了什么是0分期利息，说明它不需要支付利息，由华为商城承担，并提到调整后的宣传名称。这里的关键点是定义、功能和政策变化。\n\n第二部分提到了可以参与活动的商品，指出仅支持部分单品，合并支付不支持，后续会开放更多商品。需要强调支持的商品类型和规则。\n\n第三部分讲确认订单分期成功的方法，即选择分期支付并显示0分期利息信息。这里要明确步骤和操作方式。\n\n第四部分解释了为什么提交后没有0分期利息，指出不能与其他商品同时购买，并且分期数固定。需要说明限制条件和影响因素。\n\n第五部分询问小程序是否支持0分期利息，回答是华为商城小程序暂不支持。最后总结所有要点，确保信息完整无遗漏。\n\n现在检查是否有重复或冗余的信息，比如政策变化和活动规则可能需要合并。同时，保持语言简洁明了，符合用户要求的总结格式。\n</think>\n\n以下是文档内容的总结：\n\n**一、0分期利息定义与调整**  \n“0分期利息”指通过花呗、招行掌上生活等平台购物时无需支付利息的功能，由华为商城承担。自2023年起，宣传名称从“免息”改为“0分期利息”，依据中国银保监会政策要求，明确展示分期费用项目及计费规则。\n\n**二、支持商品与活动规则**  \n商城仅支持部分单品参与0分期利息活动，合并支付不支持。后续将逐步开放更多商品，以支付页面为准。\n\n**三、确认订单分期成功**  \n提交订单后选择分期支付并显示0分期利息信息后完成支付。\n\n**四、0分期利息限制条件**  \n若订单包含支持0分期的商品，且与其他商品同时购买，则无法享受0分期利息。需将0分期商品与非该商品分开单独购买，并明确分期数（如6期）。\n\n**五、小程序功能说明**  \n华为商城小程序暂不提供0分期利息服务。'),
 Document(metadata={'doc_id': 'd38e43f8-4da4-4a57-b611-645961742fa0'}, page_co

In [ ]:
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [ ]:
sub_docs = retriever.vectorstore.similarity_search("众测活动是否有参与限制？")
sub_docs

[Document(metadata={'doc_id': 'd38e43f8-4da4-4a57-b611-645961742fa0'}, page_content='<think>\n好的，用户让我总结提供的文档内容。首先，我需要通读整个文档，理解每个部分的信息。\n\n文档分为整体介绍、优势、适用范围、参与方式和常见问题几个部分。整体介绍里提到了众测的定义、优势以及适用范围。然后是参与方式，包括入口位置、上新频次和活动时间。常见问题也列出了来源、价格优惠、质量稳定性、发货时间、支付时效、订单查询方式、取消订单的时间和次数限制。\n\n用户的需求是总结这些内容，可能需要一个简洁明了的摘要。我需要确保涵盖所有关键点：定义、优势、适用范围、参与方式以及常见问题。同时要注意术语的一致性，比如“众测”、“低价试销”等。还要检查是否有重复的信息，并合并相关内容，使总结更流畅。\n\n另外，用户可能希望这个总结用于宣传材料或内部信息，所以需要保持专业但易懂的语气。确保每个部分的信息准确无误，没有遗漏重要细节。最后，确认总结是否全面，覆盖所有用户提到的内容点。\n</think>\n\n**众测活动总结：**\n\n1. **活动定义**  \n   以低价试销形式收集评价、销量等数据，用于测试市场反应和优化销售策略。\n\n2. **优势**  \n   - 价格优惠（对标618/双十一）；  \n   - 质量与正品一致；\n\n3. **适用范围**  \n   华为商城所有产品。\n\n4. **参与方式**  \n   - 入口：搜索“众测”进入；  \n   - 上新频次：每周一至周五更新，部分商品调整；  \n   - 活动时间：10-20天（热销商品可能延期5-10天）。\n\n5. **常见问题**  \n   - 商品来源：华为商城新品及爆款；  \n   - 价格优惠：对标618/双十一；  \n   - 质量稳定性：正品产品；  \n   - 支付与发货：24小时付款，发货前可取消订单；  \n   - 订单查询与退款：通过APP或WAP版查询，3-5个工作日退款。'),
 Document(metadata={'doc_id': 'f274cb58-67f5-46ba-add6-8e4804f79f13'}, page_content='<think

## 假设查询
LLM 还可用于生成针对特定文档可能提出的假设问题列表。然后可以嵌入这些问题


In [ ]:
sub_docs = []
for i, doc in enumerate(docs):
    _id = doc_ids[i]
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    sub_docs.extend(_sub_docs)
sub_docs

Created a chunk of size 224, which is longer than the specified 100
Created a chunk of size 117, which is longer than the specified 100


[Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f274cb58-67f5-46ba-add6-8e4804f79f13'}, page_content='一、什么是0分期利息\n\n您好，“0分期利息”是指买家使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付分期利息的功能，分期利息由华为商城承担。'),
 Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f274cb58-67f5-46ba-add6-8e4804f79f13'}, page_content='注：自2023年起，商城将相关宣传将“免息”调整为“0分期利息”，主要基于中国银保监会、中国人民银行《关于进一步促进信用卡业务规范健康发展的通知》（银保监规〔2022〕13号），要求“银行业金融机构应当在分期业务合同（协议）首页和业务办理页面以明显方式展示分期业务可能产生的所有息费项目、年化利率水平和息费计算方式。向客户展示分期业务收取的资金使用成本时，应当统一采用利息形式，并明确相应的计息规则，不得采用手续费等形式，法律法规另有规定的除外。”'),
 Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f274cb58-67f5-46ba-add6-8e4804f79f13'}, page_content='二、可以参与0分期利息活动的商品\n\n商城目前仅支持部分单品参与0分期利息，若多商品（含不支持0分期利息）合并支付则不支持0分期利息，以支付页面为准，后续会逐渐开放更多商品，敬请关注。'),
 Document(metadata={'source': '../02-agent/txt/faq-4359.txt', 'doc_id': 'f274cb58-67f5-46ba-add6-8e4804f79f13'}, page_content='三、确认订单分期成功\n\n订单提交成功后在支付方式页面选择分期支付，点选显示0分期利息的支付方式及具体0分期利息期数后，完成支付。'),
 Document(

In [ ]:
# 根据上面的文档，生成3个相关问题和回答。响应以json列表的结构返回。返回的结构参考如下
from langchain_core.output_parsers import JsonOutputParser
promptStr = '''
```
{doc}
```
根据上面的文档，生成3个相关问题和回答。响应以json列表的结构返回。返回的结构参考如下
```
[
{{"question":"问题1","answer":"回答1"}},
{{"question":"问题2","answer":"回答2"}},
{{"question":"问题3","answer":"回答3"}}
]
```
'''

prompt = ChatPromptTemplate.from_template(promptStr)

chain = (
    {"doc": lambda x: x.page_content}
    | prompt
    | model
    | JsonOutputParser()
)

In [ ]:
hypothetical_questions = chain.batch(sub_docs, {"max_concurrency": 5})

OutputParserException: Invalid json output: <think>
好的，用户让我根据提供的文档生成三个相关的问题和回答。首先，我需要仔细阅读用户给的文档内容。

文档里提到“0分期利息”是指买家使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付分期利息的功能，而分期利息由华为商城承担。所以问题应该围绕这个概念展开。

首先，可能的问题包括：什么是0分期利息？以及它如何影响用户？然后第三个问题可能是关于使用这些分期工具的注意事项或者相关功能。

接下来要确保每个回答准确，并且符合文档的信息。比如第一个问题可以直接引用定义，第二个可以问使用场景，第三个可能涉及费用或承担方的变化。需要检查是否有遗漏的关键点，比如华为商城承担的情况是否正确，避免错误信息。
</think>

[
  {
    "question": "什么是0分期利息？",
    "answer": "0分期利息是指买家在使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付额外的分期利息的功能。"
  },
  {
    "question": "该功能如何影响用户？",
    "answer": "该功能意味着用户无需承担额外的手续费，直接享受分期付款服务，但需明确说明实际费用由华为商城承担。"
  },
  {
    "question": "使用0分期利息时需要注意什么？",
    "answer": "用户应确保选择支持该功能的分期平台，并了解实际支付费用由商家或平台承担，避免因误解而产生额外负担。"
  }
]
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [ ]:
hypothetical_questions

NameError: name 'hypothetical_questions' is not defined

In [ ]:
documents = []
for item in hypothetical_questions:
    for obj in item:
        content = "问：{}\n答：{}".format(obj['question'],obj['answer'])
        documents.append(Document(page_content=content))
documents

[Document(page_content='问：什么是0分期利息？\n答：0分期利息是指买家使用花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物时无需支付分期利息的功能，分期利息由华为商城承担。'),
 Document(page_content='问：哪些渠道支持0分期利息？\n答：支持0分期利息的渠道包括花呗、招行掌上生活、工行信用卡、银联信用卡等其他分期购物。'),
 Document(page_content='问：0分期利息由谁承担？\n答：0分期利息由华为商城承担。'),
 Document(page_content="问：为什么商城将相关宣传从'免息'调整为'0分期利息'？\n答：主要基于中国银保监会、中国人民银行《关于进一步促进信用卡业务规范健康发展的通知》（银保监规〔2022〕13号），要求银行业金融机构在分期业务合同和业务办理页面以明显方式展示所有息费项目、年化利率水平和息费计算方式。向客户展示分期业务收取的资金使用成本时，应当统一采用利息形式，并明确相应的计息规则，不得采用手续费等形式，法律法规另有规定的除外。"),
 Document(page_content="问：调整后的宣传方式有什么变化？\n答：调整后，商城将相关宣传从'免息'调整为'0分期利息'，主要变化是在展示分期业务收取的资金使用成本时，统一采用利息形式，并明确相应的计息规则。"),
 Document(page_content='问：调整后的宣传方式有什么优势？\n答：调整后的宣传方式更加清晰明了地展示了分期业务的资金使用成本和年化利率水平，有利于客户更清楚地了解相关费用和风险，保障客户的知情权和选择权。'),
 Document(page_content='问：哪些商品可以参与0分期利息活动？\n答：目前仅支持部分单品参与0分期利息，后续会逐渐开放更多商品。'),
 Document(page_content='问：如果购买多件商品，其中一件不支持0分期利息，是否会影响其他商品的0分期利息资格？\n答：是的，若多商品（含不支持0分期利息）合并支付则不支持0分期利息。'),
 Document(page_content='问：如何查看哪些商品可以参与0分期利息活动？\n答：在支付页面可以查看是否支持0分期利息。'),
 Document(page_content='

In [ ]:
# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="Question", embedding_function=embeddings,persist_directory="./vector_store")
# The storage layer for the parent documents
store = InMemoryByteStore()
id_key = "doc_id"
# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

In [ ]:
retriever.vectorstore.add_documents(documents)


NameError: name 'documents' is not defined

In [ ]:
retriever.vectorstore.similarity_search("众测商品多久发货呢？")[0]

Document(page_content='问：众测商品买下后多久发货？\n答：您好，请以商品页显示为准。')